# Lovable Backtest — Laboratorio Python

Cuaderno Colab que reproduce las 6 estrategias del dashboard TypeScript.
Usa Colab (o Kaggle) para no cargar tu PC. Ver `README.md` para el modelo
de trabajo (Modelo B: Python = lab, Dashboard = producción).

## 1. Setup — instalar deps y cargar la librería

In [ ]:
# Setup robusto Colab/local: clona o actualiza el repo con reintentos y fallback a zip.
# Puedes correr esta celda cuantas veces quieras; es idempotente.
%pip install -q numpy pandas joblib matplotlib

import os, sys, shutil, subprocess, urllib.request, zipfile, io, time

REPO_URL   = "https://github.com/mrAltair432/session-whispers-flow.git"
ZIP_URL    = "https://codeload.github.com/mrAltair432/session-whispers-flow/zip/refs/heads/main"
REPO_DIR   = "/content/session-whispers-flow"
PY_DIR     = f"{REPO_DIR}/python"
TARGET_PY  = f"{PY_DIR}/lovable_backtest.py"

def _has_py():
    return os.path.isfile(TARGET_PY)

def _try_clone():
    # Si existe pero no es git válido, bórralo para clonar limpio.
    if os.path.isdir(REPO_DIR) and not os.path.isdir(os.path.join(REPO_DIR, ".git")):
        shutil.rmtree(REPO_DIR, ignore_errors=True)
    if os.path.isdir(os.path.join(REPO_DIR, ".git")):
        r = subprocess.run(["git","-C",REPO_DIR,"pull","--ff-only"],
                           capture_output=True, text=True)
        return r.returncode == 0, r.stderr
    r = subprocess.run(["git","clone","--depth","1",REPO_URL,REPO_DIR],
                       capture_output=True, text=True)
    return r.returncode == 0, r.stderr

def _try_zip():
    print("Descargando zip como fallback...")
    with urllib.request.urlopen(ZIP_URL, timeout=60) as resp:
        data = resp.read()
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    with zipfile.ZipFile(io.BytesIO(data)) as zf:
        zf.extractall("/content")
    extracted = "/content/session-whispers-flow-main"
    if os.path.isdir(extracted):
        os.rename(extracted, REPO_DIR)

ok = False
for i in range(3):
    ok, err = _try_clone()
    if ok and _has_py():
        break
    print(f"Intento git {i+1} falló: {err.strip()[:200]}")
    time.sleep(2)

if not _has_py():
    try:
        _try_zip()
    except Exception as e:
        print("Fallback zip falló:", e)

if not _has_py():
    raise FileNotFoundError(
        "No pude obtener lovable_backtest.py. Revisa conexión de Colab a GitHub "
        "(Runtime > Restart runtime) y vuelve a correr esta celda."
    )

if PY_DIR not in sys.path:
    sys.path.insert(0, PY_DIR)
os.chdir(PY_DIR)
print("Working dir:", os.getcwd())

import importlib, lovable_backtest as lb
importlib.reload(lb)
import pandas as pd, numpy as np, json
print("Estrategias disponibles:", list(lb.STRATEGIES.keys()))


## 2. Cargar CSV

Sube tus CSV de MT5 al notebook (mismos que usas en el dashboard). Se aceptan
dos formatos: `YYYY.MM.DD HH:MM,O,H,L,C,V` (script MT5) o `MM/DD/YYYY HH:MM,O,H,L,C,V` (Investing/Dukascopy).

Si solo tienes M1, `load_bars` agrega M5/M15/H1/H4 automáticamente.

In [ ]:
# Ajusta rutas: puede ser solo {'M1': 'xauusd_m1.csv'} o el set completo.
csv_files = {
    'M1':  'data/xauusd_m1.csv',
    # 'M5':  'data/xauusd_m5.csv',
    # 'M15': 'data/xauusd_m15.csv',
    # 'H1':  'data/xauusd_h1.csv',
    # 'H4':  'data/xauusd_h4.csv',
}
bars = lb.load_bars(csv_files, build_missing=True)
for tf, df in bars.items():
    print(f'{tf}: {len(df):>7} velas · desde {pd.to_datetime(df.time.min(), unit="s")} hasta {pd.to_datetime(df.time.max(), unit="s")}')

## 3. Backtest simple (sanity check)

In [ ]:
engine = 'fibo_scalping'  # prueba: fibo_scalping, gold_scalping, ema_cross_m1, straddle_breakout, smc_london, ny_continuation
res = lb.run_backtest_bars(bars, engine)
print(json.dumps(res['metrics'], indent=2, default=str))
df_trades = lb.trades_to_df(res['trades'])
df_trades.head()

## 4. Grid optimizer (paralelo con joblib)

El grid vive en `strategies_spec.json`. Puedes editarlo o pasar uno propio.

In [ ]:
spec_path = os.path.join(PY_DIR, 'strategies_spec.json')
spec = json.load(open(spec_path))
grid = spec['engines'][engine]['grid']
print('Grid:', grid)
df_grid = lb.grid_search(bars, engine, grid, min_trades=10, n_jobs=-1)
df_grid.head(10)

## 5. Walk-forward (train N meses / test M meses)

Rolling window: optimiza en train y evalúa OOS en test. La media de las
métricas OOS es lo que realmente importa (no la mejor combinación in-sample).

In [ ]:
df_wf = lb.walk_forward(bars, engine, grid, train_months=3, test_months=1, min_trades=10, n_jobs=-1)
print('Ventanas OOS:', len(df_wf))
print('Winrate OOS media :', df_wf['oos_winrate'].mean() if len(df_wf) else 'n/a')
print('Expectancy OOS media:', df_wf['oos_avg_r'].mean() if len(df_wf) else 'n/a')
df_wf

## 6. Export → best_params.json

Consumido por el dashboard TS y (más adelante) por el EA MT5 vía la tabla `mt5_signals`.

In [ ]:
# Ejemplo: optimizar TODAS las estrategias y exportar el mejor de cada una.
# Autosuficiente: recarga spec por si no corriste la celda 4.
spec = json.load(open(os.path.join(PY_DIR, 'strategies_spec.json')))
results = {}
for key, engine_def in lb.STRATEGIES.items():
    g = spec['engines'][key]['grid']
    try:
        df = lb.grid_search(bars, key, g, min_trades=10, n_jobs=-1)
        if df.empty: continue
        top = df.iloc[0]
        params = {k: (int(top[k]) if isinstance(top[k], (np.integer,)) else float(top[k]) if isinstance(top[k], np.floating) else top[k]) for k in g.keys()}
        res = lb.run_backtest_bars(bars, key, params)
        results[key] = res
        print(f'{key:22} best={params} n={res["metrics"]["trades"]:4d} avgR={res["metrics"]["avg_r"]:.3f}')
    except Exception as e:
        print(f'{key}: error → {e}')

lb.export_best_params(results, 'best_params.json')
print('\n✓ escrito best_params.json — arrástralo al dashboard.')

## 7. Filtro ML — entrenar clasificador por estrategia y exportar `ml_filters.json`

Toma los trades del backtest, entrena un modelo (LogReg + RandomForest) que
predice `p(win)` a partir del vector `features`, hace validación cruzada
temporal y busca el umbral que **maximiza la expectancy** (no solo winrate).
Exporta `ml_filters.json` con los coeficientes del LogReg + threshold óptimo
para cada engine — el dashboard puede cargarlo y filtrar señales en vivo:
solo se toman si `p(win) >= threshold`.


In [ ]:
# Filtro ML por estrategia — entrena, valida y exporta ml_filters.json
%pip install -q scikit-learn

import numpy as np, pandas as pd, json, os, math
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

FEATURE_NAMES = [
    "h4Trend","h1Sweep","m15Fvg","m15Bos","killzone","atr","h1Alignment",
    "totalScore","biasLong","hourSin","hourCos","dowSin","dowCos",
]
MIN_TRADES_ML = 40         # umbral mínimo para intentar entrenar
CV_SPLITS     = 5

def _pick_threshold(p_win, r_values):
    """Barrido de umbrales; elige el que maximiza expectancy (E[R]) con >=15 trades."""
    best = {"threshold": 0.5, "expectancy": -1e9, "trades": 0, "winrate": 0.0}
    for thr in np.linspace(0.30, 0.80, 51):
        mask = p_win >= thr
        n = int(mask.sum())
        if n < 15:
            continue
        exp = float(r_values[mask].mean())
        wr  = float((r_values[mask] > 0).mean())
        if exp > best["expectancy"]:
            best = {"threshold": float(thr), "expectancy": exp, "trades": n, "winrate": wr}
    return best

def _train_one(df_trades: pd.DataFrame):
    X_cols = [c for c in df_trades.columns if c.startswith("f_")]
    if not X_cols or len(df_trades) < MIN_TRADES_ML:
        return None
    df = df_trades.sort_values("entry_time").reset_index(drop=True)
    X = df[X_cols].values.astype(float)
    r = df["r"].values.astype(float)
    y = (r > 0).astype(int)
    if y.sum() < 5 or (len(y) - y.sum()) < 5:
        return None  # clases degeneradas

    # CV temporal para p(win) OOS
    tscv = TimeSeriesSplit(n_splits=min(CV_SPLITS, max(2, len(y)//15)))
    p_oos = np.full(len(y), np.nan)
    aucs = []
    for tr, te in tscv.split(X):
        sc = StandardScaler().fit(X[tr])
        m = LogisticRegression(max_iter=1000, C=1.0, class_weight="balanced")
        m.fit(sc.transform(X[tr]), y[tr])
        p = m.predict_proba(sc.transform(X[te]))[:, 1]
        p_oos[te] = p
        if len(set(y[te])) == 2:
            aucs.append(roc_auc_score(y[te], p))
    mask = ~np.isnan(p_oos)
    if mask.sum() < 20:
        return None

    # Modelo final sobre todo (para exportar coeficientes)
    scaler = StandardScaler().fit(X)
    lr = LogisticRegression(max_iter=1000, C=1.0, class_weight="balanced")
    lr.fit(scaler.transform(X), y)

    # RandomForest como comparador + feature importance
    rf = RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42,
                                class_weight="balanced", n_jobs=-1)
    rf.fit(X, y)

    thr = _pick_threshold(p_oos[mask], r[mask])
    base_exp = float(r.mean()); base_wr = float(y.mean())

    return {
        "n_trades": int(len(y)),
        "auc_oos": float(np.mean(aucs)) if aucs else None,
        "baseline": {"expectancy": base_exp, "winrate": base_wr, "trades": int(len(y))},
        "filtered": thr,
        "uplift_expectancy_R": thr["expectancy"] - base_exp,
        "logreg": {
            "features": FEATURE_NAMES[:len(X_cols)],
            "mean":  scaler.mean_.tolist(),
            "scale": scaler.scale_.tolist(),
            "coef":  lr.coef_[0].tolist(),
            "intercept": float(lr.intercept_[0]),
        },
        "rf_importance": dict(zip(FEATURE_NAMES[:len(X_cols)], rf.feature_importances_.round(4).tolist())),
    }

# --- 1) Corre backtest de todas las estrategias con params default -----------
all_results = {}
for key in lb.STRATEGIES.keys():
    try:
        res = lb.run_backtest_bars(bars, key)
        if res["trades"]:
            all_results[key] = res
            print(f"{key:22} {len(res['trades']):4d} trades  avgR={res['metrics']['avg_r']:+.3f}")
    except Exception as e:
        print(f"{key}: skip ({e})")

# --- 2) Entrena filtro por estrategia ----------------------------------------
ml_filters = {"_version": 1, "_feature_order": FEATURE_NAMES, "engines": {}}
summary = []
for key, res in all_results.items():
    df_t = lb.trades_to_df(res["trades"])
    out = _train_one(df_t)
    if out is None:
        print(f"{key:22} → sin datos suficientes para ML"); continue
    ml_filters["engines"][key] = out
    summary.append({
        "engine": key, "n": out["n_trades"], "auc": out["auc_oos"],
        "base_wr": out["baseline"]["winrate"], "base_expR": out["baseline"]["expectancy"],
        "thr": out["filtered"]["threshold"],
        "filt_wr": out["filtered"]["winrate"], "filt_expR": out["filtered"]["expectancy"],
        "filt_n": out["filtered"]["trades"], "uplift_R": out["uplift_expectancy_R"],
    })

df_summary = pd.DataFrame(summary)
print("\n=== Resumen ML por estrategia ===")
print(df_summary.to_string(index=False, float_format=lambda x: f"{x:+.3f}"))

# --- 3) Exporta JSON consumible por el dashboard -----------------------------
out_path = "ml_filters.json"
with open(out_path, "w") as f:
    json.dump(ml_filters, f, indent=2)
print(f"\n✓ escrito {out_path} — subir al dashboard junto a best_params.json")
print("   El dashboard aplica: p_win = sigmoid(intercept + Σ coef_i * (feat_i - mean_i)/scale_i)")
print("   y solo abre trade si p_win >= threshold del engine.")
